In [1]:
import pandas as pd
import getpass
from huggingface_hub import notebook_login
import os

from datasets import load_dataset
from fireworks.client import Fireworks
from pydantic import BaseModel, Field
from transformers import AutoTokenizer, AutoModelForCausalLM
import json
from openai import OpenAI

In [ ]:
instruction = """
          you are an expert translator between English and Danish

          #user will provide you sentences in English 
          #translate users english sentence to Danish.
          #you can only use users english sentence
          
  """

def translate_english_to_danish(english_samples_csv_file, model):
    list_of_danish_sentences = list()
    df = pd.read_csv(english_samples_csv_file)
    for i, row in enumerate(df.iterrows()):
        danish_sentence = client.chat.completions.create(
            model=model,
            messages=[
              {"role": "system", "content": instruction},
              {"role": "user", "content": row[1]['English']}
            ],
        )
        response = danish_sentence.choices[0].message.content
        list_of_danish_sentences.append(response)    
    return list_of_danish_sentences

    


In [ ]:
# instruction = """
#         you are an expert translator between English and Danish

#         #user will only give you samples of a sentence translated from English to Danish#
#         # Give the translation a score on a scale from one to ten#
#         #format should be something like 'score 1'
#         #Think through your reasoning step-by-step and write the score at the end.#
          
#   """


instruction = """
        you are an expert translator between English and Danish

        #user will only give you samples of a sentence translated from English to Danish#
        # Give the translation a score on a scale from one to ten#
        #format should only be your score          
  """


def evaluate_danish_sentences(english_danish_open_csv, evaluation_sentences):
    scores = list()
    df = pd.read_csv(english_danish_open_csv)
    df["Danish_llama_3"] = evaluation_sentences

    for i, row in enumerate(df.iterrows()): 
        
        response = openai_client.chat.completions.create(
            messages=[
                {"role": "system", "content": instruction},
                {"role": "user", "content": f'''English:{row[1]['English']} 
                                                Danish:{row[1]['Danish_llama_3']}'''}
            ],
            model="gpt-4o",

        )
        try:
            response = response.choices[0].message.content
            #score = int(json.loads(response.split('\n')[-1])['Score'])  
            scores.append(int(response))
        except json.JSONDecodeError as jde:
            continue

    return sum(scores) / len(scores)


llama_8b_avg_score = evaluate_danish_sentences("./english-danish-openai.csv", llama_8b_translation)

# print(f"Llama3 8B: {round(llama_8b_avg_score, 2)}")
print(llama_8b_avg_score)
